In this part, our goal is to predict the culinary region of each recipe based on the spices it uses. We will compare two models — **Logistic Regression and Random Forest** — to evaluate how well spices alone can distinguish between regional cuisines.

In [1]:
# Connection to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Imports
import json
import pandas as pd
import re
import itertools
import numpy as np

from IPython.display import display

# for the models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_curve,
    average_precision_score, classification_report
)
from sklearn.preprocessing import MultiLabelBinarizer, label_binarize

# for the plots
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

First, we will load the recipes dataset and spice mapping.

In [12]:
# ---------------- LOAD DATA ----------------
# Recipes dataset
with open('/content/drive/MyDrive/Data Mining - Final Project/Visualization 3 - Models/whats-cooking/train.json') as f:
    recipes = json.load(f)

# Spices mapping
with open('/content/drive/MyDrive/Data Mining - Final Project/spices json/spices_map.json') as f:
    spice_map = json.load(f)

# Convert recipes to DataFrame
df = pd.DataFrame(recipes)
print(f"Number of recipes in the dataset: {len(df)}\n")

# Display the dataset
display(
    df.head()
      .style
      .hide(axis="index")
      .set_properties(**{'text-align': 'left'})
      .set_table_styles([
          {'selector': 'th',
           'props': [('font-weight', 'bold'),
                     ('text-align', 'left')]}
      ])
)

Number of recipes in the dataset: 39774



id,cuisine,ingredients
10259,greek,"['romaine lettuce', 'black olives', 'grape tomatoes', 'garlic', 'pepper', 'purple onion', 'seasoning', 'garbanzo beans', 'feta cheese crumbles']"
25693,southern_us,"['plain flour', 'ground pepper', 'salt', 'tomatoes', 'ground black pepper', 'thyme', 'eggs', 'green tomatoes', 'yellow corn meal', 'milk', 'vegetable oil']"
20130,filipino,"['eggs', 'pepper', 'salt', 'mayonaise', 'cooking oil', 'green chilies', 'grilled chicken breasts', 'garlic powder', 'yellow onion', 'soy sauce', 'butter', 'chicken livers']"
22213,indian,"['water', 'vegetable oil', 'wheat', 'salt']"
13162,indian,"['black pepper', 'shallots', 'cornflour', 'cayenne pepper', 'onions', 'garlic paste', 'milk', 'butter', 'salt', 'lemon juice', 'water', 'chili powder', 'passata', 'oil', 'ground cumin', 'boneless chicken skinless thigh', 'garam masala', 'double cream', 'natural yogurt', 'bay leaf']"


We will clean and normalize the ingredient text, expand spice synonyms, and build regex-based rules to accurately detect spices in each recipe, accounting for variations such as plural forms and special cases (e.g., garlic cloves and pepper types), similar to what we did for the recipes we crawled from the Allrecipes website.

In [20]:
# ---------------- TEXT NORMALIZATION ----------------
def _norm_text(s: str) -> str:
    """Normalize text: lowercase, remove punctuation, unify spaces"""
    s = s.lower()
    s = re.sub(r"[-_/]", " ", s)           # replace separators (-, _, /) with space
    s = re.sub(r"[^\w\s]", " ", s)         # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()     # collapse multiple spaces into one
    return s


# ---------- HELPERS ----------
def _forms(s: str):
    """Generate common word forms (plurals, ies/ves, hyphen variants)."""
    s = s.strip().lower()
    out = {s}
    if " " in s: out.add(s.replace(" ", "-"))
    if "-" in s: out.add(s.replace("-", " "))
    if s.endswith("y") and not s.endswith(("ay","ey","iy","oy","uy")):
        out.add(s[:-1]+"ies")
    if s.endswith("f"): out.add(s[:-1]+"ves")
    if s.endswith("fe"): out.add(s[:-2]+"ves")
    if s.endswith("o") and not s.endswith(("oo","eo")):
        out.add(s+"es")
    if not s.endswith(("s","es","ies","ves")):
        out.add(s+"s")
    return out


# Expand synonyms
SPICE_MAP = {}
for canon, syns in spice_map.items():
    syns = syns or []
    base_syns = [canon, *syns]
    expanded = set(itertools.chain.from_iterable(_forms(x) for x in base_syns))
    SPICE_MAP[canon] = expanded

# Reverse lookup
SYN2CANON = {syn: canon for canon, syns in SPICE_MAP.items() for syn in syns}


# ---------- REGEX PATTERNS ----------
def _pat(term: str):
    """Create a regex pattern for a given spice synonym, allowing spaces, hyphens, or slashes as separators."""
    parts = [p for p in re.split(r"[\s\-/]+", term) if p]
    parts_esc = [re.escape(p) for p in parts]
    sep = r"(?:\s|[-/])+"  # allow spaces/hyphens/slashes
    return re.compile(r"(?<!\w)" + sep.join(parts_esc) + r"(?!\w)", re.I)

SPICE_PATTERNS = {syn: _pat(syn) for syn in SYN2CANON}
SPICE_PATTERNS["pepper"] = _pat("pepper")
SPICE_PATTERNS["peppers"] = _pat("peppers")


# ---------- SPECIAL CASES ----------
CLV_G_AFTER = re.compile(r'(\bcloves?\b)(?:\s+\w+){0,3}\s+garlic\b', re.I)
CLV_G_BEFORE = re.compile(r'\bgarlic\b(?:\s+\w+){0,3}\s+(cloves?\b)', re.I)

def _ranges_overlap(a, b):
    """Check if two index ranges overlap."""
    return max(a[0], b[0]) < min(a[1], b[1])

def _find_adjacent_pepper(low_text: str, match_start: int):
    """Resolve bare 'pepper' by looking for an adjective immediately before it
    (e.g., 'black pepper'). If none exists, fallback to black pepper."""
    tokens = re.findall(r"\w+", low_text[:match_start])
    skip_mods = {"ground","powder","finely","coarsely","fresh","freshly","dried",
             "crushed","whole","to","of","a","the","and","or"}
    meaningful = [t for t in tokens[-6:] if t not in skip_mods]
    for n in (2,1):
        if len(meaningful) >= n:
            cand_adj = " ".join(meaningful[-n:])
            candidate = f"{cand_adj} pepper"
            if candidate in SYN2CANON:
                return SYN2CANON[candidate]
    return SYN2CANON.get("black pepper") if not meaningful else None

def _accept_clove_match(matched_syn, low_text, span):
    """Return False if a 'clove' match actually refers to garlic cloves, True otherwise."""
    if matched_syn not in {"clove","cloves"}:
        return True
    a = CLV_G_AFTER.search(low_text)
    if a and _ranges_overlap(span, a.span(1)):
        return False
    b = CLV_G_BEFORE.search(low_text)
    if b and _ranges_overlap(span, b.span(1)):
        return False
    return True


# ---------- MAIN DETECTION ----------
def get_spices_from_ingredients(ingredients):
    """
    Extract and return a set of canonical spice names detected in the given ingredient list.
    Handles normalization, synonym matching, and special cases like different pepper types
    and garlic cloves.
    """
    detected = set()

    for ing in ingredients:
        low = _norm_text(ing)

        for syn, pat in SPICE_PATTERNS.items():
            for m in pat.finditer(low):
                span = m.span()
                syn_match = m.group(0).lower().strip()

                # Special case: pepper
                if syn_match in {"pepper","peppers"}:
                    resolved = _find_adjacent_pepper(low, span[0])
                    if resolved:
                        detected.add(resolved)
                    continue

                # Map to canonical
                canon = SYN2CANON.get(syn_match)
                if not canon:
                    continue

                # Special case: garlic cloves
                if not _accept_clove_match(syn_match, low, span):
                    continue

                detected.add(canon)

    return detected


# Apply spice detection to each recipe
df['detected_spices'] = df['ingredients'].apply(get_spices_from_ingredients)

# Display the cuisine and detected_spices for each recipe
display(
    df[['cuisine', 'detected_spices']].head()
      .style
      .hide(axis="index")
      .set_caption("\nDetected spices:")
      .set_properties(**{'text-align': 'left'})
      .set_table_styles([
          {'selector': 'caption',
           'props': [('color', 'black'),
                     ('font-size', '16px'),
                     ('font-weight', 'bold'),
                     ('text-align', 'left')]},
          {'selector': 'th',
           'props': [('font-weight', 'bold'),
                     ('text-align', 'left')]}
      ])
      .format({'detected_spices': lambda s: "{}" if not s else "{" + ", ".join(sorted(s)) + "}"})
)

cuisine,detected_spices
greek,{black pepper}
southern_us,"{black pepper, thyme}"
filipino,"{black pepper, garlic powder}"
indian,{}
indian,"{bay leaf, black pepper, cayenne pepper, chili powder, cumin, garam masala}"


sanity check:

Next, we count the number of detected spices per recipe to understand the coverage of our detection logic. We check what percentage of recipes have zero detected spices and display random samples for verification and sanity checking.

In [35]:
df['n_spices'] = df['detected_spices'].str.len()

def styled_display(df, cols, caption):
    """Display a DataFrame with consistent styling and optional vertical space via <br>."""
    display(
        df[cols]
          .style
          .hide(axis="index")
          .set_caption(f"<br><br>{caption}")
          .set_properties(**{'text-align': 'left'})
          .set_table_styles([
              {'selector': 'caption',
               'props': [('color', 'black'),
                         ('font-size', '16px'),
                         ('font-weight', 'bold'),
                         ('text-align', 'left')]},
              {'selector': 'th',
               'props': [('font-weight', 'bold'),
                         ('text-align', 'left')]}
          ])
    )

# % of recipes with zero detected spices
zero_spice_pct = (df['n_spices'] == 0).mean()
print(f"% recipes with 0 detected spices: {zero_spice_pct:.1%}")

# ---- Recipes with 0 detected spices ----
df_no_spices = df[df['n_spices'] == 0].sample(5, random_state=42)
styled_display(df_no_spices, ['cuisine', 'ingredients', 'n_spices'], "Recipes with 0 Detected Spices")

# ---- Random sample for sanity check ----
df_sample = df.sample(5, random_state=42)
styled_display(df_sample, ['cuisine', 'ingredients', 'detected_spices', 'n_spices'], "Random Sample for Sanity Check")

# ---- Top 5 most frequent spices ----
from collections import Counter
freq = Counter(sp for s in df['detected_spices'] for sp in s)
df_top_spices = pd.DataFrame(freq.most_common(5), columns=['Spice', 'Count'])
styled_display(df_top_spices, ['Spice', 'Count'], "Top 5 Most Frequent Detected Spices")

% recipes with 0 detected spices: 20.3%


cuisine,ingredients,n_spices
french,"['pear tomatoes', 'green beans', 'lettuce leaves', 'vinaigrette', 'fresh asparagus', 'cherry tomatoes', 'yellow bell pepper', 'cucumber', 'new potatoes', 'baby carrots']",0
southern_us,"['baking soda', 'white sugar', 'water', 'salt', 'peanuts', 'softened butter', 'light corn syrup']",0
chinese,"['dark soy sauce', 'warm water', 'vinegar', 'vegetable oil', 'corn starch', 'green bell pepper', 'ketchup', 'pork tenderloin', 'oil', 'sugar', 'water', 'flour', 'carrots', 'pineapple chunks', 'soy sauce', 'egg whites', 'salt', 'red bell pepper']",0
italian,"['unsalted butter', 'strawberries', 'sugar', 'crumbs', 'large eggs', 'cream cheese', 'mascarpone', 'balsamic vinegar']",0
mexican,"['lasagna noodles', 'ground beef', 'taco seasoning mix', 'prepar salsa', 'Mexican cheese blend', 'salsa', 'cottage cheese', 'grated parmesan cheese']",0


cuisine,ingredients,detected_spices,n_spices
chinese,"['pork', 'cooking oil', 'bamboo shoots', 'chinese rice wine', 'chinese black mushrooms', 'napa cabbage', 'sugar', 'ground black pepper', 'corn starch', 'chicken broth', 'soy sauce', 'rice cakes']",{'black pepper'},1
spanish,"['hog casings', 'hungarian paprika', 'ancho powder', 'rioja', 'minced garlic', 'pork shoulder butt', 'spanish paprika', 'olive oil', 'smoked sweet Spanish paprika', 'fatback', 'kosher salt', 'ground black pepper', 'dry red wine']","{'paprika', 'black pepper', 'ancho chili pepper'}",3
greek,"['lamb stock', 'lemon', 'lamb shoulder', 'onions', 'ground cinnamon', 'olive oil', 'dry red wine', 'bay leaf', 'pepper', 'fresh green bean', 'salt', 'dried oregano', 'tomato sauce', 'chopped tomatoes', 'garlic', 'fresh parsley']","{'cinnamon', 'black pepper', 'oregano', 'parsley', 'bay leaf'}",5
indian,"['green peas', 'cinnamon sticks', 'clove', 'chopped onion', 'cold water', 'salt', 'basmati rice', 'vegetable oil', 'cardamom pods']","{'cardamom', 'clove', 'cinnamon'}",3
italian,"['vegetable oil spray', 'cumin seed', 'grated parmesan cheese']",{'cumin'},1


Spice,Count
black pepper,13577
coriander,7084
ginger,5272
cumin,4497
parsley,4039


After we have checked that the results makes sense and that spice identification works well, we would like to divide the spices into regions according to culinary similarity. To do this, we first review the unique cuisine values in the dataset and then assign them to logical regional groups.

In [36]:
print("\n === Unique cuisines ===\n")
print(sorted(set(df['cuisine'])))
print(f"\n number of unique cuisines: {len(set(df['cuisine']))}")


 === Unique cuisines ===

['brazilian', 'british', 'cajun_creole', 'chinese', 'filipino', 'french', 'greek', 'indian', 'irish', 'italian', 'jamaican', 'japanese', 'korean', 'mexican', 'moroccan', 'russian', 'southern_us', 'spanish', 'thai', 'vietnamese']

 number of unique cuisines: 20


In [8]:
# ---------------- REGION MAPPING ----------------
# Map cuisines into broader regions
region_to_cuisines = {
    "europe": ["italian","greek","french","spanish","british","irish","russian"],
    "caribbean_and_latin_america": ["mexican","brazilian","southern_us","jamaican","cajun_creole"],
    "east_asian": ["chinese","japanese","korean","thai","vietnamese","filipino"],
    "south_asian": ["indian"],
    "north_african_middle_eastern": ["moroccan"]
}

# Reverse mapping: cuisine to region
cuisine_to_region = {
    cuisine: region
    for region, cuisines in region_to_cuisines.items()
    for cuisine in cuisines
}

# Add region column
df['region'] = df['cuisine'].map(cuisine_to_region)

# Keep only recipes where at least one spice was detected
df_with_spices = df[df['n_spices'] != 0]

# Show distribution
print("\nRecipes per region:\n")
print(df_with_spices["region"].value_counts().to_string())

# ---------------- SAVE IN CSV ----------------
# Convert the detected_spices set to a comma-separated string
df_with_spices_copy = df_with_spices.copy()
df_with_spices_copy['spices_list'] = df_with_spices_copy['detected_spices'].apply(lambda s: ', '.join(sorted(s)))

# Select and rename columns
df_to_save = df_with_spices_copy[['id', 'ingredients', 'spices_list', 'cuisine', 'region']].rename(
    columns={'id': "recipe_id"}
)

# Save to CSV
df_to_save.to_csv('/content/drive/MyDrive/Data Mining - Final Project/Visualization 3 - Models/recipes_with_spices.csv', index=False)


Recipes per region:
region
europe                          11404
caribbean_and_latin_america     10090
east_asian                       6629
south_asian                      2805
north_african_middle_eastern      770


Now, we will represent each recipe as a binary feature vector indicating which spices appear in it.

In [9]:
# ---------------- FEATURE MATRIX ----------------
# Convert spice sets into binary feature matrix
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df_with_spices['detected_spices'])  # features: one column per spice
y = df_with_spices['region']  # target: region labels

print("Feature matrix shape:", X.shape)

Feature matrix shape: (31698, 83)


Next, we will train the classification models with region labels as the target, and split the data into 80% training and 20% test sets.

We also compute a simple baseline using the majority class for comparison.

In [11]:
# ---------------- TRAIN/TEST SPLIT ----------------
# Train/Test split (stratify to maintain a ratio between the cuisine)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------- SAVE IN CSV ----------------
# ---------------- Train ----------------
train_indices = y_train.index
df_train_full = df_with_spices.loc[train_indices].copy()

df_train_full['spices_list'] = df_train_full['detected_spices'].apply(lambda s: ', '.join(sorted(s)))

# Select and rename columns
df_train_full = df_train_full[['id', 'ingredients', 'spices_list', 'cuisine', 'region']].rename(
    columns={'id': "recipe_id"}
)

# Save Train in csv
df_train_full.to_csv(
    '/content/drive/MyDrive/Data Mining - Final Project/Visualization 3 - Models/train_with_labels.csv',
    index=False
)


# ---------------- Test ----------------
test_indices = y_test.index
df_test_full = df_with_spices.loc[test_indices].copy()

# Select and rename columns
df_test_full['spices_list'] = df_test_full['detected_spices'].apply(lambda s: ', '.join(sorted(s)))
df_test_full = df_test_full[['id', 'ingredients', 'spices_list', 'cuisine', 'region']].rename(
    columns={'id': "recipe_id"}
)

# The Test with the cuisine and region (to check actual results)
df_test_full[['recipe_id', 'ingredients', 'spices_list', 'cuisine', 'region']].to_csv(
    '/content/drive/MyDrive/Data Mining - Final Project/Visualization 3 - Models/test_with_labels.csv',
    index=False
)

# The Test without the cuisine and region (what is sent to the model for prediction)
df_test_full[['recipe_id', 'ingredients', 'spices_list']].to_csv(
    '/content/drive/MyDrive/Data Mining - Final Project/Visualization 3 - Models/test_no_labels.csv',
    index=False
)

# ---------------- BASELINE ----------------
# Majority class baseline: always predict the most common region
majority_class = y_train.value_counts().idxmax()
baseline_preds = [majority_class] * len(y_test)

baseline_acc = accuracy_score(y_test, baseline_preds)

print("\nBaseline (majority class) Results:")
print(f"Majority class: {majority_class}")
print(f"Baseline Accuracy: {baseline_acc:.2f}")


Baseline (majority class) Results:
Majority class: europe
Baseline Accuracy: 0.36


To evaluate the models, we use in several metrics:

**Accuracy** – the proportion of correctly classified recipes.

**ROC AUC** (macro-averaged, one-vs-rest) – measures the model’s ability to distinguish between regions across all classes.

**Classification reports** – showing precision, recall, and F1-score for each region individually, providing insight into class-specific strengths and weaknesses.

In [9]:
# ---------------- MODELS ----------------
# Define models
models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
    ("Random Forest", RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced'))
]

# Train & Evaluate models
metrics = []

for name, model in models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)  # predicted class labels
    y_score = model.predict_proba(X_test) # predicted probabilities per class

    # Multi-class ROC AUC: use one-vs-rest and macro-average
    auc = roc_auc_score(pd.get_dummies(y_test), y_score, average='macro')

    # Classification report as dict
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    metrics.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 3),
        'ROC AUC': round(auc, 3),
        'Macro Precision': round(report['macro avg']['precision'], 3),
        'Macro Recall': round(report['macro avg']['recall'], 3),
        'Macro F1': round(report['macro avg']['f1-score'], 3),
        'Weighted Precision': round(report['weighted avg']['precision'], 3),
        'Weighted Recall': round(report['weighted avg']['recall'], 3),
        'Weighted F1': round(report['weighted avg']['f1-score'], 3),
    })

    print(f"\nClassification Report for {name}:")
    print(classification_report(y_test, y_pred, zero_division=0))

# Show comparison table
results = pd.DataFrame(metrics)
print("\nOverall Results:")
print(results[['Model', 'Accuracy', 'ROC AUC']])


Classification Report for Logistic Regression:
                              precision    recall  f1-score   support

 caribbean_and_latin_america       0.70      0.64      0.67      2018
                  east_asian       0.89      0.73      0.80      1326
                      europe       0.72      0.76      0.74      2281
north_african_middle_eastern       0.28      0.79      0.41       154
                 south_asian       0.81      0.78      0.79       561

                    accuracy                           0.72      6340
                   macro avg       0.68      0.74      0.68      6340
                weighted avg       0.75      0.72      0.73      6340


Classification Report for Random Forest:
                              precision    recall  f1-score   support

 caribbean_and_latin_america       0.74      0.61      0.67      2018
                  east_asian       0.87      0.73      0.79      1326
                      europe       0.71      0.82      0.76      2

In [10]:
# ---------------- RESULTS TABLE ----------------
results = pd.DataFrame(metrics)

display(
    results.style
    .set_caption("Model Comparison: Accuracy, ROC AUC and F1 Scores")
    .set_table_styles([
        {'selector': 'caption',
         'props': [('color', 'black'),
                   ('font-size', '16px'),
                   ('font-weight', 'bold'),
                   ('text-align', 'center')]}
    ])
    .format(precision=3)
    .hide(axis="index")
)

Model,Accuracy,ROC AUC,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Logistic Regression,0.721,0.925,0.679,0.741,0.684,0.746,0.721,0.729
Random Forest,0.733,0.917,0.686,0.727,0.695,0.746,0.733,0.734


To further evaluate model performance, we will plot Precision–Recall curves for each region in a one-vs-rest setting.

In [11]:
# ---------------- PRECISION-RECALL CURVES ----------------
# Binarize labels for one-vs-rest
classes = sorted(y_test.unique())
y_test_bin = label_binarize(y_test, classes=classes)

# Fixed color mapping for each region
color_map = {
    "europe": "#636EFA",
    "caribbean_and_latin_america": "#00CC96",
    "east_asian": "#EF553B",
    "south_asian": "#AB63FA",
    "north_african_middle_eastern": "#FFA15A"
}

# Create subplot
fig = make_subplots(
    rows=1, cols=len(models),
    subplot_titles=[name for name, _ in models],
    shared_yaxes=True
)

# Store AP scores
ap_scores = {name: {} for name, _ in models}

# Track which classes already appeared in the legend
shown_in_legend = set()

for col, (name, model) in enumerate(models, start=1):
    y_score = model.predict_proba(X_test)

    for i, class_name in enumerate(classes):
        precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
        ap = average_precision_score(y_test_bin[:, i], y_score[:, i])

        # Save AP
        ap_scores[name][class_name] = ap

        # Show legend only for first occurrence of each class
        show_legend = class_name not in shown_in_legend
        shown_in_legend.add(class_name)

        # Add curve
        fig.add_trace(
            go.Scatter(
                x=recall,
                y=precision,
                mode='lines',
                name=f"{class_name}",
                legendgroup=class_name,
                showlegend=show_legend,
                line=dict(color=color_map[class_name])
            ),
            row=1, col=col
        )

# Layout
fig.update_layout(
    title_text="<b>Precision–Recall Curves Comparison</b>",
    xaxis_title="Recall",
    yaxis_title="Precision",
    template="plotly_white",
    width=1200,
    height=600,
    legend=dict(
        font=dict(size=14)
    )
)

fig.show()


# Convert AP scores dictionary into a DataFrame
ap_df = pd.DataFrame(ap_scores).T  # models as rows, classes as columns
ap_df = ap_df[classes]  # keep columns in the same order as classes

print("\nAverage Precision (AP) scores:")
display(ap_df.round(3))


Average Precision (AP) scores:


,caribbean_and_latin_america,east_asian,europe,north_african_middle_eastern,south_asian
Logistic Regression,0.739,0.858,0.825,0.684,0.820
Random Forest,0.749,0.853,0.824,0.532,0.805


This visualization highlights which regions are easier or harder to distinguish. The average precision (AP) score, shown in the legend, summarizes the area under each PR curve and allows for a direct comparison across regions.

We want to know intuitively how many of the observations the model correctly predicted in each region.
The following plot visualizes the Random Forest model’s performance per cuisine region. For each region, we compare the total number of recipes in the test set to the number of recipes correctly predicted by the model.

In [12]:
# ---------------- Random Forest predictions ----------------
y_pred_rf = models[1][1].predict(X_test)

# Create DataFrame comparing true regions vs correct predictions
df_comparison = pd.DataFrame({
    "Region": y_test,
    "Is_Correct": y_pred_rf == y_test
})

# Summarize counts per region
df_summary = df_comparison.groupby("Region")["Is_Correct"].agg(
    Total_Recipes="size",
    Correct_Predictions="sum"
).reset_index()

# Compute success percentage
df_summary["Success_pct"] = (df_summary["Correct_Predictions"] / df_summary["Total_Recipes"] * 100).round(1)

# Prepare data for grouped bar chart
df_plot = df_summary.melt(
    id_vars=["Region", "Success_pct"],
    value_vars=["Total_Recipes", "Correct_Predictions"],
    var_name="Type",
    value_name="Count"
)

# Color mapping
color_map = {
    "Total_Recipes": "#b2df8a",
    "Correct_Predictions": "#33a02c"
}

# ---------------- Plot ----------------
fig = px.bar(
    df_plot,
    x="Region",
    y="Count",
    color="Type",
    barmode="group",
    text=df_plot["Count"].apply(lambda x: f"<b>{x}</b>"),
    color_discrete_map=color_map,
    category_orders={"Type": ["Total_Recipes", "Correct_Predictions"]},
    labels={"Count": "Number of Recipes", "Region": "Cuisine Region"}
)

# "Correct_Predictions" numbers inside bars
for trace in fig.data:
    if trace.name == "Correct_Predictions":
        trace.textposition = "inside"
        trace.textfont = dict(size=10)

# Add percentage labels above each pair of bars
for i, row in df_summary.iterrows():
    top_bar_height = max(row["Total_Recipes"], row["Correct_Predictions"])
    fig.add_annotation(
        x=row["Region"],
        y=top_bar_height + 60,  # slightly above the taller bar
        text=f"<b>{row['Success_pct']}%</b>",
        showarrow=False,
        font=dict(color="black", size=14),
        xanchor="center"
    )

# ---------------- Layout Improvements ----------------
fig.update_layout(
    title=dict(text="<b>Random Forest: True vs Correctly Predicted Recipes per Region</b>",
               x=0.5),
    title_font=dict(size=20, family="Arial", color="black"),
    xaxis_title="Cuisine Region",
    yaxis_title="Number of Recipes",
    yaxis=dict(gridcolor='lightgray', zeroline=True),
    legend=dict(title=None,   # remove legend title
                x=0.85, y=1, # position top-right
                bordercolor='black',
                borderwidth=1
               ),
    template="plotly_white",
    uniformtext_minsize=10,
    uniformtext_mode='show',
    bargap=0.3,  # space between grouped bars
)

fig.show()

This section creates an interactive web interface using Gradio, allowing enter a list of spices from a recipe, select a trained model (Logistic Regression or Random Forest), and see the predicted cuisine region based on the entered spices.

In [13]:
!pip install -q gradio

In [14]:
import gradio as gr

# ---------------- Function to run prediction ----------------
def predict_with_model(model_name, spices_text):
    """
    Predict Cuisine's Region based on recipe's spices using the selected model.

    Parameters:
    - model_name: str, either "Logistic Regression" or "Random Forest"
    - spices_text: str, comma-separated spices (user input)

    Returns:
    - str, prediction and confidence
    """
    # Select model
    if model_name == "Logistic Regression":
        model = models[0][1]
    else:
        model = models[1][1]

    # ---------------- Preprocessing ----------------
    # Split input string into a list
    input_list = [s.strip() for s in spices_text.split(",") if s.strip()]

    # Detect canonical spices using your pipeline
    detected = get_spices_from_ingredients(input_list)

    if not detected:
        return "No valid spices detected in input! Please enter recognized spices."

    # Convert to model features (binary vector using same MultiLabelBinarizer as training)
    sample = mlb.transform([detected])   # shape (1, n_spices)

    # ---------------- Prediction ----------------
    pred = model.predict(sample)[0]
    prob = model.predict_proba(sample).max()

    return f"""
    Prediction with **{model_name}**:<br><br>
    <span style="background-color:rgba(0, 128, 0, 0.3); color:white; padding:8px 12px; border-radius:6px; font-weight:bold; display:inline-block;">
        {pred}
    </span>
    <br><br>
    **Model confidence:** {prob:.1%}
    """


# ---------------- Build Gradio Interface ----------------
with gr.Blocks() as demo:
    # Header
    gr.Markdown(
        """
        # Cuisine's Region Prediction by Spices 🍲
        Enter a list of recipe's spices and select a model to predict the cuisine's region.
        """
    )

    # Spices input
    spices_box = gr.Textbox(
        label="Spices (comma separated)",
        value="chili powder, garlic, cumin",
        placeholder="e.g. basil, oregano, tomato",
        lines=2
    )

    # Buttons row
    with gr.Row():
        btn_lr = gr.Button("Run Logistic Regression", variant="primary")
        btn_rf = gr.Button("Run Random Forest", variant="primary")

    # Output box
    output = gr.Markdown("### Prediction Result")

    # Bind buttons
    btn_lr.click(
        fn=lambda ing: predict_with_model("Logistic Regression", ing),
        inputs=spices_box,
        outputs=output
    )
    btn_rf.click(
        fn=lambda ing: predict_with_model("Random Forest", ing),
        inputs=spices_box,
        outputs=output
    )

# Launch the app
demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>